In [1]:
from pydantic import BaseModel
from typing import List, Optional

class Message(BaseModel):
    role: str
    content: str

class ArchInputState(BaseModel):
    current_sequence: str
    context: str

class ArchOutputState(BaseModel):
    next_component: Optional[str]
    explanation: Optional[str]
    current_sequence: Optional[str]
    reasoning: Optional[str]

class StructuredArchitectureState(BaseModel):
    messages: List[Message]
    input_state: ArchInputState
    output_state: Optional[ArchOutputState]
    is_complete: bool
    iteration_count: int


In [2]:
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()

# Verify API keys are set
required_keys = ['GROQ_API_KEY', 'LANGSMITH_API_KEY', 'TAVILY_API_KEY']
for key in required_keys:
    if not os.getenv(key):
        raise ValueError(f"Environment variable {key} is not set")

In [3]:
from groq import Groq


client = Groq(api_key=os.environ["GROQ_API_KEY"])

def call_groq(prompt: str) -> str:
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.1,
        max_tokens=700
    )
    return response.choices[0].message.content


In [4]:
def iteration_prompt(current_sequence: str, context: str) -> str:
    return f"""
You are a senior distributed systems architect.

Current architecture sequence
{current_sequence}

System context
{context}

Task
Predict the next most logical architectural component.

Rules
Return ONLY valid JSON
Add exactly one component
Append it to the sequence
Stop when the system is complete

JSON schema
{{
  "next_component": "string or null",
  "explanation": "string",
  "updated_sequence": "string",
  "reasoning": "string",
  "complete": boolean
}}
"""


In [5]:
import json
# from state import StructuredArchitectureState, ArchOutputState, Message
# from llm import call_groq
# from prompts import iteration_prompt

def architecture_step(state: StructuredArchitectureState) -> StructuredArchitectureState:
    prompt = iteration_prompt(
        state.input_state.current_sequence,
        state.input_state.context
    )

    raw = call_groq(prompt)

    try:
        parsed = json.loads(raw)
    except Exception:
        state.messages.append(Message(role="ai", content="INVALID_JSON"))
        state.is_complete = True
        return state

    state.iteration_count += 1

    state.output_state = ArchOutputState(
        next_component=parsed.get("next_component"),
        explanation=parsed.get("explanation"),
        current_sequence=parsed.get("updated_sequence"),
        reasoning=parsed.get("reasoning")
    )

    state.input_state.current_sequence = parsed.get("updated_sequence")
    state.is_complete = parsed.get("complete", False)

    state.messages.append(Message(role="ai", content=raw))

    return state


In [6]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
# from state import StructuredArchitectureState
# from nodes import architecture_step

builder = StateGraph(StructuredArchitectureState)

builder.add_node("architect", architecture_step)

builder.set_entry_point("architect")

builder.add_conditional_edges(
    "architect",
    lambda state: END if state.is_complete or state.iteration_count >= 15 else "architect"
)

memory = MemorySaver()
graph = builder.compile(checkpointer=memory)


In [7]:
# from graph import graph
# from state import StructuredArchitectureState, ArchInputState, Message

initial_state = StructuredArchitectureState(
    messages=[Message(role="human", content="Design a distributed job scheduler")],
    input_state=ArchInputState(
        current_sequence="user request",
        context="distributed job scheduler for batch processing with retries and autoscaling"
    ),
    output_state=None,
    is_complete=False,
    iteration_count=0
)

config = {
    "configurable": {"thread_id": "job_scheduler_test"},
    "recursion_limit": 15
}

result = graph.invoke(initial_state, config)

print("Final sequence")
print(json.dumps(result, indent=4, default=str))
print(result["input_state"])
print()

print("Iterations", result["iteration_count"])
print()

print("Components predicted")
for m in result["messages"]:
    if m.role == "ai":
        print(m.content)


Final sequence
{
    "messages": [
        "role='human' content='Design a distributed job scheduler'",
        "role='ai' content='{\\n  \"next_component\": \"API Gateway\",\\n  \"explanation\": \"Exposes a RESTful endpoint for clients to submit batch jobs and performs authentication, rate limiting, and request validation before forwarding to the scheduler.\",\\n  \"updated_sequence\": \"user request -> API Gateway\",\\n  \"reasoning\": \"After a user initiates a request, the system needs a front\u2011door component that can accept external traffic, enforce security policies, and translate the request into an internal job submission format. An API Gateway is the logical next step before queuing or scheduling the job.\",\\n  \"complete\": false\\n}'",
        "role='ai' content='{\\n  \"next_component\": \"Job Submission Service\",\\n  \"explanation\": \"Handles request validation, authentication, job definition creation, and enqueues jobs for processing.\",\\n  \"updated_sequence\": \

In [70]:
import os
import operator
from typing import List, Annotated, Optional, TypedDict
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from groq import Groq
import instructor
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()

# --- 1. Define Structures ---

class ArchitectureStep(BaseModel):
    """The specific output we want from the LLM."""
    next_component: Optional[str] = Field(..., description="The single next component to add (e.g. 'Load Balancer'). Return None if complete.")
    reasoning: str = Field(..., description="Why this component comes next.")
    is_complete: bool = Field(False, description="Set to True if the system architecture is fully defined.")

# --- 2. Define Graph State ---

# In LangGraph, use TypedDict for state to make updates cleaner
class AgentState(TypedDict):
    messages: Annotated[List[str], operator.add] # 'add' reducer appends to list automatically
    current_sequence: str
    context: str
    architecture_steps: List[ArchitectureStep]
    is_complete: bool
    iteration_count: int

# --- 3. Setup LLM Client with Instructor ---

# Patch the Groq client to support strict Pydantic response_model
client = instructor.from_groq(
    Groq(api_key=os.environ["GROQ_API_KEY"]),
    mode=instructor.Mode.JSON
)

def get_architecture_prediction(sequence: str, context: str) -> ArchitectureStep:
    """Calls Groq and forces a valid Pydantic object response."""
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile", # UPDATED: Valid Groq model
        response_model=ArchitectureStep,
        messages=[
            {
                "role": "system", 
                "content": "You are a distributed systems architect. Build the system one component at a time."
            },
            {
                "role": "user", 
                "content": f"""
                Context: {context}
                Current Sequence: {sequence}
                
                Task: Identify the ONE next logical component.
                If the system is functionally complete, set is_complete to True.
                """
            }
        ],
        temperature=0.1,
    )
    return response

# --- 4. Define Nodes ---

def architect_node(state: AgentState):
    # 1. Get prediction
    step = get_architecture_prediction(state["current_sequence"], state["context"])
    
    # 2. Update logic (Python side, not LLM side)
    new_sequence = state["current_sequence"]
    
    if step.next_component:
        if state["current_sequence"] == "Start":
             new_sequence = step.next_component
        else:
             new_sequence = f"{state['current_sequence']} -> {step.next_component}"
    
    # 3. Create update dictionary (Don't mutate state in place!)
    return {
        "architecture_steps": [step], # Appends because of list type, or we handle manual aggregation if not reduced
        "current_sequence": new_sequence,
        "is_complete": step.is_complete,
        "iteration_count": state["iteration_count"] + 1,
        "messages": [f"Added: {step.next_component} (Reason: {step.reasoning})"]
    }

# --- 5. Build Graph ---

workflow = StateGraph(AgentState)

workflow.add_node("architect", architect_node)
workflow.set_entry_point("architect")

def should_continue(state: AgentState):
    if state["is_complete"]:
        return END
    if state["iteration_count"] >= 10:
        return END
    return "architect"

workflow.add_conditional_edges("architect", should_continue)

graph = workflow.compile(checkpointer=MemorySaver())

# --- 6. Execution ---

initial_state = {
    "messages": [],
    "current_sequence": "Start",
    "context": "A distributed video transcoding pipeline (Upload -> Process -> Storage)",
    "architecture_steps": [],
    "is_complete": False,
    "iteration_count": 0
}

config = {"configurable": {"thread_id": "1"}}

# Run the graph
for event in graph.stream(initial_state, config):
    for key, value in event.items():
        print(f"\n--- Node: {key} ---")
        # Print the latest update to the sequence
        if "current_sequence" in value:
            print(f"Sequence: {value['current_sequence']}")


--- Node: architect ---
Sequence: Load Balancer

--- Node: architect ---
Sequence: Load Balancer -> Upload Server

--- Node: architect ---
Sequence: Load Balancer -> Upload Server -> Transcoding Server

--- Node: architect ---
Sequence: Load Balancer -> Upload Server -> Transcoding Server -> Storage Server

--- Node: architect ---
Sequence: Load Balancer -> Upload Server -> Transcoding Server -> Storage Server


In [71]:
# human in loop

import os
import operator
from typing import List, Annotated, Optional, TypedDict, Literal
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from groq import Groq
import instructor
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

# Load env variables
load_dotenv()

# --- 1. Setup Models & LLM ---

class ArchitecturePrediction(BaseModel):
    """Output for Option 1: AI Auto-suggest"""
    next_component: str = Field(..., description="The single next component to add.")
    reasoning: str = Field(..., description="Why this component comes next.")

class ValidationResult(BaseModel):
    """Output for Custom Input: Checking user's idea"""
    is_valid: bool = Field(..., description="Is this component technically feasible and logical here?")
    feedback: str = Field(..., description="If valid, why it fits. If invalid, why it fails.")

client = instructor.from_groq(
    Groq(api_key=os.environ["GROQ_API_KEY"]),
    mode=instructor.Mode.JSON
)

# --- 2. Define State ---

class AgentState(TypedDict):
    # The architecture string (e.g., "Client -> Load Balancer")
    current_sequence: str
    context: str
    
    # User interaction data
    user_input: Optional[str]
    system_message: Optional[str] # To hold errors or success messages
    
    # History log
    history: Annotated[List[str], operator.add]

# --- 3. Define Nodes ---

def human_node(state: AgentState):
    """
    This node stops and asks the user for input.
    """
    print("\n" + "="*50)
    print(f"CURRENT ARCHITECTURE:\n{state['current_sequence']}")
    print("="*50)
    
    if state.get("system_message"):
        print(f"System Message: {state['system_message']}")
    
    print("\nOptions:")
    print(" [1] Auto-generate next step (Default)")
    print(" [2] Stop and Finish")
    print(" [Type Name] Enter a custom component (e.g., 'Redis Cache')")
    
    user_choice = input("\nYour choice: ").strip()
    
    # Default to 1 if empty
    if not user_choice:
        user_choice = "1"
        
    return {"user_input": user_choice, "system_message": None}

def ai_generate_node(state: AgentState):
    """
    Option 1: AI predicts the next component.
    """
    print("🤖 AI is thinking...")
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        response_model=ArchitecturePrediction,
        messages=[
            {"role": "system", "content": "You are a distributed systems architect."},
            {"role": "user", "content": f"Context: {state['context']}\nCurrent: {state['current_sequence']}\nTask: Predict next component."}
        ]
    )
    
    new_fragment = f" -> {response.next_component}"
    updated_sequence = state['current_sequence'] + new_fragment
    
    return {
        "current_sequence": updated_sequence,
        "history": [f"AI Added: {response.next_component}"],
        "system_message": f"Auto-added '{response.next_component}' based on: {response.reasoning}"
    }

def validate_input_node(state: AgentState):
    """
    Option Custom: AI checks if user input is valid.
    """
    user_component = state["user_input"]
    print(f"🔍 Validating '{user_component}'...")
    
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        response_model=ValidationResult,
        messages=[
            {"role": "system", "content": "You are a strict technical reviewer."},
            {"role": "user", "content": f"""
             Context: {state['context']}
             Current Sequence: {state['current_sequence']}
             User Suggestion: {user_component}
             
             Task: Determine if the user suggestion is a valid next step.
             """}
        ]
    )
    
    if response.is_valid:
        new_fragment = f" -> {user_component}"
        return {
            "current_sequence": state['current_sequence'] + new_fragment,
            "history": [f"User Added: {user_component}"],
            "system_message": f"✅ Accepted '{user_component}'. {response.feedback}"
        }
    else:
        # Do not update sequence, just send feedback
        return {
            "system_message": f"❌ Rejected '{user_component}'. {response.feedback}"
        }

# --- 4. Define Logic Branching ---

def router(state: AgentState) -> Literal["ai_generate", "validate_input", "end"]:
    choice = state["user_input"]
    
    if choice == "1":
        return "ai_generate"
    elif choice == "2":
        return "end"
    else:
        return "validate_input"

# --- 5. Build Graph ---

workflow = StateGraph(AgentState)

workflow.add_node("human", human_node)
workflow.add_node("ai_generate", ai_generate_node)
workflow.add_node("validate_input", validate_input_node)

workflow.set_entry_point("human")

workflow.add_conditional_edges(
    "human",
    router,
    {
        "ai_generate": "ai_generate",
        "validate_input": "validate_input",
        "end": END
    }
)

# Loop back to human after processing
workflow.add_edge("ai_generate", "human")
workflow.add_edge("validate_input", "human")

graph = workflow.compile(checkpointer=MemorySaver())

# --- 6. Run ---

initial_state = {
    "current_sequence": "User Request",
    "context": "A high-traffic e-commerce checkout system",
    "history": [],
    "user_input": None,
    "system_message": "Initialization Complete."
}

config = {"configurable": {"thread_id": "session_2"}}

# We use graph.invoke (or stream) here. 
# Since we use input() inside the node, standard recursion limits apply.
try:
    graph.invoke(initial_state, config)
except Exception:
    pass # Handles the "Stop" gracefully if needed, or just let it finish naturally via END


CURRENT ARCHITECTURE:
User Request
System Message: Initialization Complete.

Options:
 [1] Auto-generate next step (Default)
 [2] Stop and Finish
 [Type Name] Enter a custom component (e.g., 'Redis Cache')
🤖 AI is thinking...

CURRENT ARCHITECTURE:
User Request -> Load Balancer
System Message: Auto-added 'Load Balancer' based on: To distribute incoming traffic efficiently and prevent single point of failure, a load balancer should be added next to handle the high traffic volume of the e-commerce checkout system.

Options:
 [1] Auto-generate next step (Default)
 [2] Stop and Finish
 [Type Name] Enter a custom component (e.g., 'Redis Cache')
🤖 AI is thinking...

CURRENT ARCHITECTURE:
User Request -> Load Balancer -> Application Server
System Message: Auto-added 'Application Server' based on: To handle the user request, the load balancer needs to direct the traffic to an application server where the checkout system's business logic can be executed, ensuring the request is processed and t